# 03 · Loss and averaging

Loss weighting λ, small architectures, SWA / EMA of weights, residual nets.

Needs `lib.py` and the `.npy` data written by `00_setup_data_feature_importance`. Comments are in Russian; outputs are kept as the record of each experiment.


In [ ]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: НЕТ GPU
overlay         108G   21G   88G  20% /
total 30G
-rw-r--r-- 1  501 staff  28G Dec 31  2019 train.parquet
-rw-r--r-- 1  501 staff  88M Dec 31  2019 valid_mask.parquet
-rw------- 1 root root  1.7G Nov  2 13:23 valid.parquet
overlay         108G   50G   59G  47% /


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU: Runtime -> Change runtime type -> T4')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: НЕТ GPU: Runtime -> Change runtime type -> T4


In [4]:
%run -i /content/drive/MyDrive/wnn/lib.py

need = {'train': (600, 0, False), 'train3k': (3000, 0, False), 'valid300': (300, 1, True)}
def done(n): return all(os.path.exists(f'/content/data/{n}_{s}.npy') for s in (['X', 'Y'] + (['M'] if n.startswith('valid') else [])))
missing = [n for n in need if not done(n)]
print('нет файлов для:', missing or 'всё на месте')
if missing:
    if not os.path.exists(f'{ROOT}/train.parquet'):
        print('качаю данные (около 11 минут)...')
        subprocess.run("curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
                       "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null",
                       shell=True, cwd='/content')
    for n in missing:
        src = 'valid' if n.startswith('valid') else 'train'
        build(src, n, *need[n])
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

нет файлов для: ['valid300']


ArrowInvalid: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xtr, Ytr = ld('train_X'), np.load('/content/data/train_Y.npy'); Y6 = Ytr          # 600 последовательностей
Y3k = np.load('/content/data/train3k_Y.npy')                                       # 3000 (X: ld('train3k_X'))
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy'); Y3 = Yva
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)
f3k = fam_of(Y3k)

paths = [f'{CKPT_DIR}/gru_w3k_best.pt', '/content/gru_w3k_best.pt', '/content/drive/MyDrive/wnn/gru_w3k_best.pt']
CKPT = next((p for p in paths if os.path.exists(p)), None)
if CKPT:
    net128 = Net(128).to(dev); net128.load_state_dict(torch.load(CKPT, map_location=dev)); net128.eval()
    print('h=128 с', CKPT, '->', mix_score(net128), '(раньше было 0.487 / 0.653)')
else:
    print('чекпойнта gru_w3k_best.pt нет. Переобучить (около 19 минут):\n'
          "  Xtr, Ytr = ld('train3k_X'), Y3k; torch.manual_seed(0); net128 = train(Net().to(dev), epochs=8, tag='gru_w3k')")

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
макс. отклонение от единицы: 0.0 | классы: [ 387  375  955 1283]
h=128 с /content/drive/MyDrive/wnn/gru_w3k_best.pt -> {'все строки': 0.488, 'is_scored': 0.654} (раньше было 0.487 / 0.653)


Продолжаем

In [ ]:
import numpy as np, time
a = np.array([0.1, -0.2], np.float32); n = 200000
t = time.time()
for _ in range(n): np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
print(f'nan_to_num: {(time.time() - t) / n * 1e6:.1f} мкс')
t = time.time()
for _ in range(n):
    ok = abs(a[0]) < 1e6 and abs(a[1]) < 1e6
print(f'сравнение abs < порога: {(time.time() - t) / n * 1e6:.2f} мкс')

nan_to_num: 16.6 мкс
сравнение abs < порога: 0.70 мкс


In [ ]:
class NetS(nn.Module):
    def __init__(s, h, layers):
        super().__init__(); s.gru = nn.GRU(NF, h, num_layers=layers, batch_first=True); s.head = nn.Linear(h, 2)
    def forward(s, x, h=None):
        y, h = s.gru(x, h); return s.head(y), h

if 'loss_orig' not in globals(): loss_orig = loss_fn
LAMC = 0.0
def loss_fn(p, y, mask):
    y = y.clamp(-2, 2); main = (y.abs() * (p - y) ** 2)[mask].mean()
    if LAMC == 0: return main
    cs = []
    for j in (0, 1):
        yy, pp = y[..., j][mask], p[..., j][mask]; w = yy.abs(); sw = w.sum() + 1e-8
        my, mp = (w * yy).sum() / sw, (w * pp).sum() / sw
        cov = (w * (yy - my) * (pp - mp)).sum(); vy, vp = (w * (yy - my) ** 2).sum(), (w * (pp - mp) ** 2).sum()
        cs.append(cov / torch.sqrt(vy * vp + 1e-8))
    return main - LAMC * torch.stack(cs).mean()

Xtr = ld('train_X'); Ytr = np.load('/content/data/train_Y.npy')                       # те же 600 последовательностей, что в эксперименте с оракулом
Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)
print('обучающих:', len(Xtr), '| валидационных:', len(Xva), '| контроль для сравнения: 0.468 / 0.6405')

variants = {'1 слой, h=64': (lambda: NetS(64, 1), 0.0), '2 слоя, h=64': (lambda: NetS(64, 2), 0.0), '2×128 + корреляция': (lambda: Net(), 0.5)}
for name, (mk, lam) in variants.items():
    LAMC = lam
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(mk().to(dev), epochs=8, tag=f'v_{seed}')
        print('>>>', name, seed, 'параметров', sum(p.numel() for p in m.parameters()), mix_score(m), flush=True)
LAMC = 0.0; loss_fn = loss_orig

обучающих: 600 | валидационных: 300 | контроль для сравнения: 0.468 / 0.6405


/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/8 | 25 с | loss 1.1417 | valid все строки 0.432 [0.431 0.432] | is_scored 0.617 [0.614 0.62 ]
эпоха 2/8 | 20 с | loss 1.0731 | valid все строки 0.448 [0.448 0.448] | is_scored 0.630 [0.629 0.631]
эпоха 3/8 | 23 с | loss 1.0408 | valid все строки 0.458 [0.459 0.458] | is_scored 0.642 [0.635 0.648]
эпоха 4/8 | 29 с | loss 1.0309 | valid все строки 0.457 [0.456 0.459] | is_scored 0.646 [0.644 0.649]
эпоха 5/8 | 30 с | loss 1.0114 | valid все строки 0.467 [0.464 0.469] | is_scored 0.652 [0.649 0.656]
эпоха 6/8 | 27 с | loss 0.9987 | valid все строки 0.462 [0.467 0.456] | is_scored 0.654 [0.648 0.66 ]
эпоха 7/8 | 27 с | loss 0.9829 | valid все строки 0.467 [0.467 0.468] | is_scored 0.655 [0.65 0.66]
эпоха 8/8 | 26 с | loss 0.9842 | valid все строки 0.467 [0.466 0.467] | is_scored 0.656 [0.651 0.662]
>>> 1 слой, h=64 0 параметров 34306 {'все строки': 0.467, 'is_scored': 0.644}
эпоха 1/8 | 28 с | loss 1.1397 | valid все строки 0.427 [0.429 0.426] | is_scored 0.612 [0.605 0.619]
эпоха 

In [ ]:
def make_loss(lam):
    def f(p, y, mask):
        y = y.clamp(-2, 2); main = (y.abs() * (p - y) ** 2)[mask].mean()
        cs = []
        for j in (0, 1):
            yy, pp = y[..., j][mask], p[..., j][mask]; w = yy.abs(); sw = w.sum() + 1e-8
            my, mp = (w * yy).sum() / sw, (w * pp).sum() / sw
            cov = (w * (yy - my) * (pp - mp)).sum(); vy, vp = (w * (yy - my) ** 2).sum(), (w * (pp - mp) ** 2).sum()
            cs.append(cov / torch.sqrt(vy * vp + 1e-8))
        return main - lam * torch.stack(cs).mean()
    return f

Xtr = ld('train_X'); Ytr = np.load('/content/data/train_Y.npy')
Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
for lam in (1.0, 2.0):
    loss_fn = make_loss(lam)
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(Net().to(dev), epochs=8, tag=f'lam_{seed}')
        print('>>> вес корреляции', lam, 'сид', seed, mix_score(m), flush=True)
loss_fn = loss_orig

эпоха 1/8 | 35 с | loss 0.8272 | valid все строки 0.443 [0.441 0.445] | is_scored 0.641 [0.633 0.649]
эпоха 2/8 | 42 с | loss 0.6715 | valid все строки 0.457 [0.463 0.451] | is_scored 0.641 [0.643 0.639]
эпоха 3/8 | 37 с | loss 0.6275 | valid все строки 0.451 [0.456 0.445] | is_scored 0.636 [0.634 0.639]
эпоха 4/8 | 37 с | loss 0.6071 | valid все строки 0.454 [0.458 0.45 ] | is_scored 0.635 [0.637 0.632]
эпоха 5/8 | 41 с | loss 0.5686 | valid все строки 0.472 [0.47  0.473] | is_scored 0.650 [0.648 0.651]
эпоха 6/8 | 38 с | loss 0.5361 | valid все строки 0.471 [0.477 0.466] | is_scored 0.657 [0.65  0.664]
эпоха 7/8 | 39 с | loss 0.4952 | valid все строки 0.473 [0.472 0.474] | is_scored 0.657 [0.652 0.663]
эпоха 8/8 | 37 с | loss 0.4861 | valid все строки 0.473 [0.472 0.474] | is_scored 0.659 [0.653 0.665]
>>> вес корреляции 1.0 сид 0 {'все строки': 0.473, 'is_scored': 0.647}
эпоха 1/8 | 36 с | loss 0.8244 | valid все строки 0.445 [0.443 0.447] | is_scored 0.640 [0.634 0.647]
эпоха 2/8 |

In [ ]:
print(os.path.exists('/content/data/train3k_X.npy'), os.path.exists('/content/data/train3k_Y.npy'))   # должно быть True True
Xtr = ld('train3k_X'); Ytr = np.load('/content/data/train3k_Y.npy')
Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
loss_fn = make_loss(2.0)
torch.manual_seed(0)
m3 = train(Net().to(dev), epochs=8, tag='lam2_3k')
print('>>> 3000 посл., вес корреляции 2.0, последняя эпоха:', mix_score(m3), '| контроль (лучшая эпоха) 0.488 / 0.654')
loss_fn = loss_orig

True True
эпоха 1/8 | 164 с | loss 0.3468 | valid все строки 0.446 [0.444 0.449] | is_scored 0.651 [0.645 0.657]
эпоха 2/8 | 162 с | loss 0.2101 | valid все строки 0.468 [0.471 0.465] | is_scored 0.650 [0.649 0.651]
эпоха 3/8 | 161 с | loss 0.1836 | valid все строки 0.464 [0.465 0.464] | is_scored 0.643 [0.64  0.645]
эпоха 4/8 | 161 с | loss 0.1544 | valid все строки 0.478 [0.477 0.479] | is_scored 0.661 [0.656 0.666]
эпоха 5/8 | 158 с | loss 0.1316 | valid все строки 0.488 [0.489 0.487] | is_scored 0.661 [0.659 0.664]
эпоха 6/8 | 157 с | loss 0.0894 | valid все строки 0.492 [0.495 0.49 ] | is_scored 0.666 [0.662 0.671]
эпоха 7/8 | 156 с | loss 0.0477 | valid все строки 0.491 [0.493 0.489] | is_scored 0.666 [0.662 0.67 ]
эпоха 8/8 | 155 с | loss 0.0265 | valid все строки 0.492 [0.493 0.492] | is_scored 0.667 [0.662 0.671]
>>> 3000 посл., вес корреляции 2.0, последняя эпоха: {'все строки': 0.492, 'is_scored': 0.654} | контроль (лучшая эпоха) 0.488 / 0.654


In [ ]:
!pip install -q onnx onnxruntime
%run -i /content/drive/MyDrive/wnn/lib.py
import importlib.util, shutil, time
from types import SimpleNamespace
BASE = '/content/wnn_connectome_starterpack/baseline'
torch.manual_seed(0); net = Net(128).eval(); z = torch.zeros(1, 1, 128)
os.makedirs('/content/bench2/nan', exist_ok=True); os.makedirs('/content/bench2/thr', exist_ok=True)
torch.onnx.export(Step(net), (torch.zeros(1, 1, 112), z, z.clone()), '/content/bench2/nan/model.onnx',
                  input_names=['features', 'hidden_0', 'hidden_1'], output_names=['prediction', 'hidden_0_out', 'hidden_1_out'],
                  opset_version=17, dynamo=False)
shutil.copy('/content/bench2/nan/model.onnx', '/content/bench2/thr/model.onnx')
src = open(f'{BASE}/solution.py').read().replace('baseline.onnx', 'model.onnx')
old = 'return np.asarray(prediction[0, 0], dtype=np.float32)'; assert old in src
open('/content/bench2/nan/solution.py', 'w').write(src.replace(old, 'return np.nan_to_num(np.asarray(prediction[0, 0], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)'))
open('/content/bench2/thr/solution.py', 'w').write(src.replace(old,
    'out = np.asarray(prediction[0, 0], dtype=np.float32)\n        if not (abs(out[0]) < 1e6 and abs(out[1]) < 1e6):\n            out = np.zeros(2, np.float32)\n        return out'))

def load_sol(path, name):
    sp = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m); return m
SOL = {'baseline': load_sol(f'{BASE}/solution.py', 'sb'), 'nan_to_num': load_sol('/content/bench2/nan/solution.py', 'sn'),
       'порог': load_sol('/content/bench2/thr/solution.py', 'st')}
X = np.clip(np.random.default_rng(0).standard_normal((4, L, NF)), -2.4, 2.4).astype(np.float32)

def speed(sol):
    mdl = sol.PredictionModel(); t = time.time()
    for s in range(len(X)):
        for st in range(L):
            mdl.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= WARM, state=X[s, st]))
    return (time.time() - t) / (len(X) * L) * 1e6

T = {k: [] for k in SOL}
for rnd in range(3):
    for k, sol in SOL.items(): T[k].append(speed(sol)); print(f'раунд {rnd + 1}: {k:11s} {T[k][-1]:.0f} мкс/строка', flush=True)
med = {k: float(np.median(v)) for k, v in T.items()}
for k in med: print(f'{k:11s} {med[k]:.0f} мкс | {med[k] / med["baseline"]:.2f}× baseline -> оценка на платформе ~{27.7 * med[k] / med["baseline"]:.0f} мин')
print('(для nan_to_num на платформе было 37 мин, для baseline 27.7)')

/tmp/ipykernel_542/1310531609.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Step(net), (torch.zeros(1, 1, 112), z, z.clone()), '/content/bench2/nan/model.onnx',
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4571: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


раунд 1: baseline    81 мкс/строка
раунд 1: nan_to_num  120 мкс/строка
раунд 1: порог       90 мкс/строка
раунд 2: baseline    81 мкс/строка
раунд 2: nan_to_num  120 мкс/строка
раунд 2: порог       94 мкс/строка
раунд 3: baseline    81 мкс/строка
раунд 3: nan_to_num  120 мкс/строка
раунд 3: порог       93 мкс/строка
baseline    81 мкс | 1.00× baseline -> оценка на платформе ~28 мин
nan_to_num  120 мкс | 1.47× baseline -> оценка на платформе ~41 мин
порог       93 мкс | 1.14× baseline -> оценка на платформе ~32 мин
(для nan_to_num на платформе было 37 мин, для baseline 27.7)


In [ ]:
import math
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xtr = ld('train3k_X'); Ytr = np.load('/content/data/train3k_Y.npy')
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)

def with_params(plist):                       # новая сеть с заданными весами
    m = Net(128).to(dev)
    with torch.no_grad():
        for p, q in zip(m.parameters(), plist): p.copy_(q)
    return m.eval()

def train_avg(epochs=8, bs=32, ch=1000, lr=2e-3, end_frac=0.1, ema_decay=0.999, swa_last=3, seed=0):
    torch.manual_seed(seed); model = Net(128).to(dev); params = list(model.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    total = epochs * (len(Xtr) // bs) * (L // ch); warm = int(0.05 * total)
    lr_at = lambda s: s / warm if s < warm else end_frac + (1 - end_frac) * 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    ema = [p.detach().clone() for p in params]; swa = [torch.zeros_like(p) for p in params]; n_swa = 0
    rng = np.random.default_rng(0)
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(len(Xtr)); run = []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev).clamp(-2, 2); h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1)
                yy = y[:, t:t + ch]; loss = (yy.abs() * (o - yy) ** 2)[mask].mean()
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); run.append(loss.item())
                with torch.no_grad():
                    torch._foreach_mul_(ema, ema_decay); torch._foreach_add_(ema, [p.detach() for p in params], alpha=1 - ema_decay)
        if ep >= epochs - swa_last:
            with torch.no_grad():
                for s, p in zip(swa, params): s.mul_(n_swa / (n_swa + 1)).add_(p.detach() / (n_swa + 1))
            n_swa += 1
        r1, r2 = evaluate(model, Xva, Yva, Mva), evaluate(with_params(ema), Xva, Yva, Mva)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | последние веса: все {r1["все строки"][0]:.3f} / is_scored {r1["is_scored"][0]:.3f} | '
              f'EMA: все {r2["все строки"][0]:.3f} / is_scored {r2["is_scored"][0]:.3f}', flush=True)
    return {'последние веса': [p.detach().clone() for p in params], 'EMA': ema, f'SWA (последние {swa_last} эпох)': swa}

W = train_avg()
for name, pl in W.items():
    print('>>>', name, mix_score(with_params(pl)), flush=True)
print('эталон: обычный рецепт, 3000 посл., лучшая эпоха: 0.488 / 0.654')
torch.save({k: [p.cpu() for p in v] for k, v in W.items()}, f'{CKPT_DIR}/avg_weights.pt')

эпоха 1/8 | 158 с | loss 1.0547 | последние веса: все 0.440 / is_scored 0.641 | EMA: все 0.469 / is_scored 0.649
эпоха 2/8 | 152 с | loss 1.0025 | последние веса: все 0.472 / is_scored 0.651 | EMA: все 0.483 / is_scored 0.664
эпоха 3/8 | 152 с | loss 0.9923 | последние веса: все 0.469 / is_scored 0.651 | EMA: все 0.486 / is_scored 0.665
эпоха 4/8 | 152 с | loss 0.9821 | последние веса: все 0.475 / is_scored 0.656 | EMA: все 0.488 / is_scored 0.666
эпоха 5/8 | 151 с | loss 0.9712 | последние веса: все 0.489 / is_scored 0.663 | EMA: все 0.490 / is_scored 0.666
эпоха 6/8 | 150 с | loss 0.9535 | последние веса: все 0.487 / is_scored 0.664 | EMA: все 0.490 / is_scored 0.667
эпоха 7/8 | 152 с | loss 0.9347 | последние веса: все 0.486 / is_scored 0.666 | EMA: все 0.489 / is_scored 0.667
эпоха 8/8 | 150 с | loss 0.9211 | последние веса: все 0.486 / is_scored 0.664 | EMA: все 0.487 / is_scored 0.666
>>> последние веса {'все строки': 0.486, 'is_scored': 0.652}
>>> EMA {'все строки': 0.487, 'is_s

In [ ]:
m = with_params(W['SWA (последние 3 эпох)'])
P = predict(m, Xva, np.arange(len(Xva)))
rng = np.random.default_rng(0); fold = np.zeros(len(Xva), int)
for f in range(4):
    ix = np.where(fam_v == f)[0]; rng.shuffle(ix); fold[ix[:len(ix) // 2]] = 1

def fit_cal(ix, scale_only):
    mask = Mva[ix] & np.broadcast_to(post, Mva[ix].shape); cal = []
    for j in (0, 1):
        y = np.clip(Yva[ix][..., j][mask], -2, 2).astype(np.float64); p = P[ix][..., j][mask].astype(np.float64); w = np.abs(y); sw = w.sum()
        my, mp = (w * y).sum() / sw, (w * p).sum() / sw
        a = (w * (y - my) * (p - mp)).sum() / (w * (p - mp) ** 2).sum()
        cal.append((a, 0.0 if scale_only else my - a * mp))
    return cal

def mix_eval(Pm, ixf):
    h = np.array([len(ixf[f]) for f in range(4)], float); h /= h.sum()
    Sm = {f: sums_np(Pm[ixf[f]], Yva[ixf[f]], Mva[ixf[f]]) for f in range(4)}
    return {k: corr(sum((VALMIX[f] / h[f]) * Sm[f][k] for f in range(4))) for k in ('все строки', 'is_scored')}

res = {'без калибровки': [], 'масштаб': [], 'масштаб + сдвиг': []}
for tr_fold in (0, 1):
    ixf_te = {f: np.where((fam_v == f) & (fold != tr_fold))[0] for f in range(4)}
    res['без калибровки'].append(mix_eval(P, ixf_te))
    for name, so in (('масштаб', True), ('масштаб + сдвиг', False)):
        P2 = P.copy()
        for f in range(4):
            cal = fit_cal(np.where((fam_v == f) & (fold == tr_fold))[0], so)
            for j in (0, 1): P2[ixf_te[f], :, j] = np.clip(cal[j][0] * P[ixf_te[f], :, j] + cal[j][1], -2, 2)
        res[name].append(mix_eval(P2, ixf_te))
for name, r in res.items():
    print(f'{name:16s} все строки {np.mean([x["все строки"] for x in r]):.3f} | is_scored {np.mean([x["is_scored"] for x in r]):.3f}   (две половины: '
          f'{r[0]["is_scored"]:.3f}, {r[1]["is_scored"]:.3f})')

без калибровки   все строки 0.488 | is_scored 0.655   (две половины: 0.672, 0.638)
масштаб          все строки 0.488 | is_scored 0.655   (две половины: 0.672, 0.639)
масштаб + сдвиг  все строки 0.488 | is_scored 0.654   (две половины: 0.670, 0.637)


In [ ]:
class NetR(nn.Module):
    def __init__(s, drop=0.0, cdrop=0.0, h=128):
        super().__init__(); s.cdrop = cdrop
        s.gru = nn.GRU(NF, h, num_layers=2, batch_first=True, dropout=drop); s.head = nn.Linear(h, 2)
    def forward(s, x, h=None):
        if s.training and s.cdrop > 0:
            x = x * (torch.rand(x.shape[0], 1, x.shape[2], device=x.device) > s.cdrop).float()
        y, h = s.gru(x, h); return s.head(y), h

def plain_loss(p, y, mask):
    y = y.clamp(-2, 2); return (y.abs() * (p - y) ** 2)[mask].mean()
loss_fn = plain_loss

Xtr = ld('train_X'); Ytr = np.load('/content/data/train_Y.npy')
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
variants = {'dropout 0.2': dict(drop=0.2), 'channel dropout 0.1': dict(cdrop=0.1), 'оба': dict(drop=0.2, cdrop=0.1)}
for name, kw in variants.items():
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(NetR(**kw).to(dev), epochs=8, tag=f'r_{seed}')
        print('>>>', name, seed, mix_score(m), '| контроль 0.468 / 0.6405', flush=True)

эпоха 1/8 | 39 с | loss 1.1478 | valid все строки 0.432 [0.436 0.428] | is_scored 0.628 [0.624 0.632]
эпоха 2/8 | 43 с | loss 1.0697 | valid все строки 0.452 [0.455 0.448] | is_scored 0.627 [0.626 0.629]
эпоха 3/8 | 41 с | loss 1.0406 | valid все строки 0.458 [0.464 0.453] | is_scored 0.640 [0.641 0.638]
эпоха 4/8 | 36 с | loss 1.0283 | valid все строки 0.452 [0.449 0.454] | is_scored 0.629 [0.632 0.627]
эпоха 5/8 | 37 с | loss 1.0052 | valid все строки 0.468 [0.47  0.467] | is_scored 0.646 [0.65  0.642]
эпоха 6/8 | 36 с | loss 0.9858 | valid все строки 0.468 [0.474 0.463] | is_scored 0.653 [0.647 0.66 ]
эпоха 7/8 | 36 с | loss 0.9612 | valid все строки 0.471 [0.471 0.47 ] | is_scored 0.655 [0.65 0.66]
эпоха 8/8 | 38 с | loss 0.9569 | valid все строки 0.469 [0.47  0.469] | is_scored 0.657 [0.651 0.662]
>>> dropout 0.2 0 {'все строки': 0.47, 'is_scored': 0.644} | контроль 0.468 / 0.6405
эпоха 1/8 | 38 с | loss 1.1444 | valid все строки 0.433 [0.436 0.429] | is_scored 0.626 [0.622 0.629]